# 03 — PDF-Grounded Multilingual Retriever Training & Evaluation
**Project**: Farmer Advisory Chatbot Using Multilingual NLP  
**Goal**: Fine-tune `intfloat/multilingual-e5-small` with MultipleNegativesRankingLoss on official agricultural extension PDFs.

### 1. Colab Setup (GitHub Clone + Google Drive for Outputs)
Clones your code and PDFs directly from GitHub, mounts Google Drive to permanently save checkpoints and benchmark results.

In [ ]:
import os, sys, shutil, torch

# 1. GPU Verification
assert torch.cuda.is_available(), 'CRITICAL ERROR: No GPU found! In Colab menu, go to Runtime > Change runtime type > T4 GPU.'
print(f'Active GPU: {torch.cuda.get_device_name(0)}')

# 2. Mount Google Drive for saving checkpoints & evaluation metrics
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/farmer_advisory_chatbot'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Checkpoints and results will be saved permanently to: {DRIVE_ROOT}')

# 3. Verify project directory
# If you opened this notebook directly in Colab without cloning yet, enter your repo URL below:
REPO_URL = 'https://github.com/REPLACE_WITH_YOUR_USERNAME/farmer_advisory_chatbot.git'
if not os.path.exists('/content/farmer_advisory_chatbot'):
    if 'REPLACE_WITH_YOUR_USERNAME' not in REPO_URL:
        !git clone {REPO_URL} /content/farmer_advisory_chatbot
        %cd /content/farmer_advisory_chatbot
    elif os.path.exists(DRIVE_ROOT) and os.path.exists(os.path.join(DRIVE_ROOT, 'train_retriever.py')):
        shutil.copytree(DRIVE_ROOT, '/content/farmer_advisory_chatbot', ignore=shutil.ignore_patterns('*.git*', '__pycache__', 'smoke_*'))
        %cd /content/farmer_advisory_chatbot
    else:
        print('Please ensure your repo is cloned to /content/farmer_advisory_chatbot or update REPO_URL above.')
else:
    %cd /content/farmer_advisory_chatbot

# Anti-smoke guard: verify real mode environment
assert os.path.exists('kb/chunks.jsonl'), 'CRITICAL: kb/chunks.jsonl not found! Ingestion needed.'

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q pymupdf pdfplumber rank-bm25 sentence-transformers transformers accelerate

### 2. PDF Ingestion & Verbatim Knowledge Base Construction
Extracts verbatim paragraphs and tables with exact document, page, and heading attribution.

In [ ]:
# Ingest official PDFs from kb/pdfs/
!python data_prep/ingest_pdfs.py --real

# Display coverage report summary
import json
with open('logs/kb_report.json', 'r') as f:
    report = json.load(f)
print(f"Total Chunks Ingested: {report['total_chunks']}")
print(f"Chunks with chemical dosages: {report['chunks_with_dosage']}")
print("Disease Coverage:", json.dumps(report['disease_coverage'], indent=2))

### 3. Multilingual Synthetic Query Generation (Qwen + NLLB)
Generates realistic farmer-style questions via `Qwen2.5-1.5B-Instruct` and translates to Hindi and Marathi using `NLLB-200-distilled-600M`.

In [ ]:
# Step 3A: Generate English queries with local LLM on T4 GPU
!python data_prep/generate_queries.py --real --backend llm --seed 42

# Step 3B: Multilingual translation (NLLB-200) to Hindi and Marathi
!python data_prep/generate_queries.py --real --backend nllb --seed 42

# Verify grouped split report (0 source_id leakage)
with open('logs/retrieval_split_report.json', 'r') as f:
    split_rep = json.load(f)
print('Retrieval Split Report:', json.dumps(split_rep, indent=2))
assert split_rep['source_id_overlap'] == 0, 'Data leakage detected across retrieval splits!'

### 4. Zero-Shot & BM25 Baseline Evaluation
Evaluate BM25 Direct, BM25 Pivot, and un-fine-tuned `multilingual-e5-small` to establish the baseline.

In [ ]:
!python eval/eval_retriever.py --real \
    --kb-path kb/chunks.jsonl \
    --test-queries data/retrieval_val.csv \
    --bootstrap-iter 500

### 5. 1-Epoch GPU Sanity Check
Runs 1 epoch on GPU to verify gradient backward pass, MNRL loss computation, and checkpoint writing.

In [ ]:
!python train_retriever.py --config configs/retriever.yaml --real --epochs 1 --batch-size 32 --output-root {DRIVE_ROOT} --seed 42

### 6. Full Fine-Tuning Run (5 Epochs)
Fine-tunes `intfloat/multilingual-e5-small` with MultipleNegativesRankingLoss and cosine learning rate decay.

In [ ]:
!python train_retriever.py --config configs/retriever.yaml --real --epochs 5 --batch-size 64 --lr 2e-5 --output-root {DRIVE_ROOT} --seed 42

### 7. Evaluation & Ablation Benchmark on Val & Human-Verified Test Sets
Benchmarks all 5 retriever configurations across languages with 95% bootstrap confidence intervals.

In [ ]:
# Benchmark on Validation Split and Human-verified Test Set
!python eval/eval_retriever.py --real \
    --model-dir {DRIVE_ROOT}/checkpoints/retriever/e5_small_finetuned_seed42_best \
    --kb-path kb/chunks.jsonl \
    --test-queries data/retrieval_val.csv \
    --bootstrap-iter 1000

# Sync results back to Google Drive
!cp -r results {DRIVE_ROOT}/
!cp -r logs/retriever {DRIVE_ROOT}/logs/
print('All checkpoints, logs, and results synchronized to Google Drive.')

### 8. Results Display & Failure Analysis

In [ ]:
import pandas as pd

print('=== RETRIEVAL BENCHMARK RESULTS ===')
df_results = pd.read_csv('results/retrieval_results.csv', comment='#')
display(df_results[['label', 'language', 'num_queries', 'recall@1', 'recall@5', 'mrr@10', 'ndcg@10', 'avg_latency_ms']])

print('\n=== RETRIEVER FAILURE ANALYSIS SAMPLE (Top 10) ===')
df_fail = pd.read_csv('results/retrieval_failure_analysis.csv', comment='#')
display(df_fail.head(10))